In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm import tqdm

links = []
regions = 4

print("Collecting house links...")
for i in tqdm(range(1, 50), desc="Pages", unit="page"):
    r = requests.get(
        f"https://iranfile.ir/properties/buy-apartment?regions=1,-11,-1"
    )
    content = r.content.decode("utf-8")
    soup = BeautifulSoup(content, "html.parser")
    for a in soup.findAll("a", href=True, attrs={"class": "grd_search_links"}):
        links.append(a["href"])
links = list(set(links))
file_path = "HomeDatasetTehranRegion4_R4_1404_12.txt"

with open(file_path, "w", encoding="utf-8") as f:
    f.write(
        "h_type\tdate\taddress\tloc\tnum_floor\tunit_per_floor\tprice\tage\tstatuse\tview\tdoc_status\tnorth\tsought\twest\teast\tfloor\tarea\tnum_sleep\ttel\tkitch\tservice\tfloor_covering\topen\tparking\twarehouse\tbalcony\tequipment\tnearby\n"
    )

print("Extracting house data...")
for item in tqdm(links, desc="Houses", unit="house"):
    try:
        r = requests.get(item)
        content = r.content.decode("utf-8")
        soup = BeautifulSoup(content, features="html.parser")
        first_values = []
        with open(file_path, "a", encoding="utf-8") as f:
            for a in soup.findAll(["div", "span"], attrs={"class": "file-data"}):
                first_values.append(a.text)

            f.write(
                first_values[2].strip()
                + "\t"
                + first_values[0].strip()
                + "\t"
                + first_values[5].strip()
                + "\t"
                + str(regions)
                + "\t"
                + first_values[6].strip()
                + "\t"
                + first_values[7].strip()
                + "\t"
                + first_values[8].strip()
                + "\t"
                + first_values[10].strip()
                + "\t"
                + first_values[11].strip()
                + "\t"
                + first_values[12].strip()
                + "\t"
                + first_values[13].strip()
                + "\t"
            )

        direction = soup.findAll("div", attrs={"class": "file-active"})
        direction_dict = {}

        for i in ["شمالی", "جنوبی", "غربی", "شرقی"]:
            direction_dict.setdefault(i, 0)
            for d in direction:
                if str(d.text).strip() == i:
                    direction_dict[i] = 1

        with open(file_path, "a", encoding="utf-8") as f:
            for i in ["شمالی", "جنوبی", "غربی", "شرقی"]:
                # north,south,west,east
                f.write(f"{direction_dict[i]}\t")
            rows = soup.findAll("td")
            # floor,area\tnum_sleep,tel,kitch,service,floor_covering
            for i in range(1, 14, 2):
                f.write(f"{rows[i].text.strip()}\t")
            # open,parking,warehouse,balcony
            for i in [15, 17, 19, 21]:
                if "file-checked" in rows[i].find().get("class"):
                    f.write("1\t")
                else:
                    f.write("0\t")
            e = []
            for i in soup.findAll("div", attrs={"class": "check-box-info"}):
                e.append(i.text.strip())
            f.write(f"{'_'.join(e)}\t")

            for i in soup.findAll("div", attrs={"class": "info-melk-file-details"}):
                f.write(f"{i.select('div:nth-of-type(2)')[0].text.strip()}\t")

            # nearby
            e = []
            for i in soup.findAll("div", attrs={"class": "form-section-header"}):
                e.append(i.text.strip())
            f.write(f"{'_'.join(e)}\n")
    except Exception as ex:
        print(f"Error: {ex} in {item}")
        continue


Pages: 100%|█████████████████████████████████████████████████████████████████████████| 49/49 [00:36<00:00,  1.33page/s]


Extracting house data...


Houses:  17%|███████████▊                                                         | 167/980 [02:52<15:51,  1.17s/house]

Error: list index out of range in https://iranfile.ir/FileDetail/10566139/فروش_82متری_آپارتمان_2خواب،در_منطقه_فرجام_غربی(_تا_شهید_باقری)


Houses:  44%|██████████████████████████████▋                                      | 436/980 [07:29<08:23,  1.08house/s]

Error: list index out of range in https://iranfile.ir/FileDetail/10491372/آپارتمان_118متری_2خواب،جهت_فروش_در_منطقه_لویزان


Houses: 100%|█████████████████████████████████████████████████████████████████████| 980/980 [17:04<00:00,  1.05s/house]


In [2]:
import os
os.getcwd()

'D:\\projects\\stock 1403'